# M-Shots Learning

In this notebook, we'll explore small prompt engineering techniques and recommendations that will help us elicit responses from the models that are better suited to our needs.

In [2]:
from openai import OpenAI
import os

from dotenv import load_dotenv, find_dotenv
_ = load_dotenv(find_dotenv()) # read local .env file

OPENAI_API_KEY  = os.getenv('OPENAI_API_KEY')

# Formatting the answer with Few Shot Samples.

To obtain the model's response in a specific format, we have various options, but one of the most convenient is to use Few-Shot Samples. This involves presenting the model with pairs of user queries and example responses.

Large models like GPT-3.5 respond well to the examples provided, adapting their response to the specified format.

Depending on the number of examples given, this technique can be referred to as:
* Zero-Shot.
* One-Shot.
* Few-Shots.

With One Shot should be enough, and it is recommended to use a maximum of six shots. It's important to remember that this information is passed in each query and occupies space in the input prompt.



In [3]:
# Function to call the model.
def return_OAIResponse(user_message, context):
    client = OpenAI(
    # This is the default and can be omitted
    api_key=OPENAI_API_KEY,
)

    newcontext = context.copy()
    newcontext.append({'role':'user', 'content':"question: " + user_message})

    response = client.chat.completions.create(
            model="gpt-3.5-turbo",
            messages=newcontext,
            temperature=1,
        )

    return (response.choices[0].message.content)

In this zero-shots prompt we obtain a correct response, but without formatting, as the model incorporates the information he wants.

In [4]:
#zero-shot
context_user = [
    {'role':'system', 'content':'You are an expert in F1.'}
]
print(return_OAIResponse("Who won the F1 2010?", context_user))

Sebastian Vettel won the Formula 1 World Championship in 2010 driving for the Red Bull Racing team.


For a model as large and good as GPT 3.5, a single shot is enough to learn the output format we expect.


In [5]:
#one-shot
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2000 f1 championship?
     Driver: Michael Schumacher.
     Team: Ferrari."""}
]
print(return_OAIResponse("Who won the F1 2011?", context_user))

Driver: Sebastian Vettel.
Team: Red Bull Racing.


Smaller models, or more complicated formats, may require more than one shot. Here a sample with two shots.

In [6]:
#Few shots
context_user = [
    {'role':'system', 'content':
     """You are an expert in F1.

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

The 2006 Formula 1 World Championship was won by Fernando Alonso, driving for the Renault F1 Team.


In [7]:
print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton.
Team: Mercedes.


We've been creating the prompt without using OpenAI's roles, and as we've seen, it worked correctly.

However, the proper way to do this is by using these roles to construct the prompt, making the model's learning process even more effective.

By not feeding it the entire prompt as if they were system commands, we enable the model to learn from a conversation, which is more realistic for it.

In [8]:
#Recomended solution
context_user = [
    {'role':'system', 'content':'You are and expert in f1.\n\n'},
    {'role':'user', 'content':'Who won the 2010 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Sebastian Vettel. \nTeam: Red Bull. \nPoints: 256. """},
    {'role':'user', 'content':'Who won the 2009 f1 championship?'},
    {'role':'assistant', 'content':"""Driver: Jenson Button. \nTeam: BrawnGP. \nPoints: 95. """},
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton. 
Team: Mercedes. 
Points: 413.


We could also address it by using a more conventional prompt, describing what we want and how we want the format.

However, it's essential to understand that in this case, the model is following instructions, whereas in the case of use shots, it is learning in real-time during inference.

In [9]:
context_user = [
    {'role':'system', 'content':"""You are and expert in f1.
    You are going to answer the question of the user giving the name of the rider,
    the name of the team and the points of the champion, following the format:
    Drive:
    Team:
    Points: """
    }
]

print(return_OAIResponse("Who won the F1 2019?", context_user))

Driver: Lewis Hamilton
Team: Mercedes
Points: 413


In [10]:
context_user = [
    {'role':'system', 'content':
     """You are classifying .

     Who won the 2010 f1 championship?
     Driver: Sebastian Vettel.
     Team: Red Bull Renault.

     Who won the 2009 f1 championship?
     Driver: Jenson Button.
     Team: BrawnGP."""}
]
print(return_OAIResponse("Who won the F1 2006?", context_user))

Driver: Fernando Alonso.  
Team: Renault.


Few Shots for classification.


In [11]:
context_user = [
    {'role':'system', 'content':
     """You are an expert in reviewing product opinions and classifying them as positive or negative.

     It fulfilled its function perfectly, I think the price is fair, I would buy it again.
     Sentiment: Positive

     It didn't work bad, but I wouldn't buy it again, maybe it's a bit expensive for what it does.
     Sentiment: Negative.

     I wouldn't know what to say, my son uses it, but he doesn't love it.
     Sentiment: Neutral
     """}
]
print(return_OAIResponse("I'm loving it.", context_user))

Sentiment: Positive


# Exercise
 - Complete the prompts similar to what we did in class. 
     - Try at least 3 versions
     - Be creative
 - Write a one page report summarizing your findings.
     - Were there variations that didn't work well? i.e., where GPT either hallucinated or wrong
 - What did you learn?

# Zero-shot

In [13]:

context_zero = [{'role': 'system', 'content': 'You are an assistant that extracts ingredients from a recipe.'}]
print("Zero-shot:")
print(return_OAIResponse("Omelet: 3 eggs, salt, pepper, butter. Put all the egg in a bowl. Add the spices.", context_zero))

Zero-shot:
Ingredients extracted from the recipe:
- 3 eggs
- Salt
- Pepper
- Butter


# Few-shot (2)

In [14]:
context_few = [
    {'role': 'system', 'content': 'You are a culinary expert that lists ingredients.'},
    {'role': 'user', 'content': 'Salad: lettuce, tomatoes, vinaigrette.'},
    {'role': 'assistant', 'content': '- Lettuce\n- Tomatoes\n- Vinaigrette'},
    {'role': 'user', 'content': 'Sandwich: bread, ham, cheese.'},
    {'role': 'assistant', 'content': '- Bread\n- Ham\n- Cheese'}
]
print("\nFew-shot:")
print(return_OAIResponse("Pasta: spaghetti, tomato sauce, basil.", context_few))


Few-shot:
- Spaghetti
- Tomato sauce
- Basil


# Few-shot JSON format

In [17]:
context_few_json = [
    {'role': 'system', 'content': 'Extract the ingredients in JSON format.'},
    {'role': 'user', 'content': 'Tacos: tortilla, meat, onion.'},
    {'role': 'assistant', 'content': '{"ingredients": ["tortilla", "meat", "onion"]}'}
]
print("\nFew-shot JSON :")
print(return_OAIResponse("Smoothie: banana, milk, strawberry.", context_few_json))


Few-shot JSON :
{"ingredients": ["banana", "milk", "strawberry"]}


### Report

**Summary of Results:**
I tested three approaches for extracting ingredients from recipes.
* **Version 1 (Zero-shot):** The model understood the task but provided the answer as a complete sentence and "-".The result was correct, it is difficult to parse for automated data pipelines.
* **Version 2 (Few-shot):** By providing two examples, the model immediately adopted the bulleted list format I expected, without needing explicit textual instructions on how to format the output.
* **Version 3 (JSON Few-shot):** This method proved to be the most robust. By forcing a JSON structure, the model conformed exactly to the requested schema, making it ideal for software integration.

**Analysis of Errors:**
During an initial trial, I attempted a short "Few-shot" prompt using inconsistent examples (e.g., one example in a list and another as a sentence). The model began to hallucinate by mixing these styles. I concluded that consistency in examples is just as critical as the number of examples provided.

**What I Learned:**
Few-shot prompting allows us to "program" the model through examples rather than just descriptions. For formatting tasks, using explicit roles (`user`/`assistant`) is often more effective than overloading the `system` message with complex rules. Furthermore, saving tokens by minimizing repetitive instructions helps optimize both latency and costs.